# OmniLSS CPU Performance Benchmark
# OmniLSS CPU 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongfangzhizhu/OmniLSS/blob/main/examples/colab/05_performance_cpu.ipynb)

This notebook benchmarks OmniLSS CPU performance against R GAMLSS across 9 distributions and 3 data sizes.

 notebook  OmniLSS  CPU  R GAMLSS  9  3 

## 📋 Contents
1. Environment setup2. Install R for comparison /  R 
3. Benchmark: 9 distributions × 3 data sizes / 9  × 3 
4. Visualize results5. Summary table
---

## 1. Environment Setup

In [ ]:
# Check environment / 检查运行环境
import sys
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab / 运行在 Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally / 运行在本地环境")

# Install OmniLSS / 安装 OmniLSS
import subprocess
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "git+https://github.com/dongfangzhizhu/OmniLSS.git#subdirectory=omnilss"], check=True)
else:
    subprocess.run(["pip", "install", "-q", "-e", "../../omnilss"], check=True)

import jax
jax.config.update("jax_enable_x64", True)
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 2. Install R for Comparison /  R 

In [ ]:
# Install R and required packages / 安装 R 和所需包
if IN_COLAB:
    print("Installing R... / 安装 R...")
    subprocess.run(["apt-get", "install", "-y", "-q", "r-base"], check=False)
    print("Installing R packages... / 安装 R 包...")
    subprocess.run(
        ["Rscript", "-e",
         "install.packages(c('gamlss','jsonlite'), repos='https://cran.r-project.org', quiet=TRUE)"],
        check=False
    )
    print("✓ R environment ready / R 环境准备完成")
else:
    print("Please ensure R, gamlss, and jsonlite are installed locally.")

## 3. Benchmark Setup

In [ ]:
import numpy as np
import time
import json
import tempfile
import csv
from pathlib import Path
from omnilss import gamlss
from omnilss.distributions import resolve_family

DISTRIBUTIONS = ["NO", "GA", "LOGNO", "PO", "BI", "NBI", "BE", "ZIP", "ZAGA"]
DATA_SIZES = [100, 1000, 10000]
N_REPEATS = 3

# R fitting script / R 拟合脚本
R_FIT_SCRIPT = '''
suppressMessages(library(gamlss))
suppressMessages(library(jsonlite))
args <- commandArgs(trailingOnly=TRUE)
data_file <- args[1]; dist <- args[2]; formula <- args[3]
df <- read.csv(data_file)
t0 <- proc.time()["elapsed"]
fit <- tryCatch(
  gamlss(as.formula(formula), family=dist, data=df, trace=FALSE),
  error=function(e) NULL
)
elapsed <- proc.time()["elapsed"] - t0
if (is.null(fit)) {
  cat(toJSON(list(success=FALSE), auto_unbox=TRUE), "\\n")
} else {
  cat(toJSON(list(success=TRUE, deviance=fit$G.deviance, r_time=elapsed), auto_unbox=TRUE), "\\n")
}
'''

def run_r_fit(dist, formula, data):
    with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as f:
        writer = csv.writer(f)
        keys = list(data.keys())
        writer.writerow(keys)
        for row in zip(*[data[k] for k in keys]):
            writer.writerow(row)
        csv_path = f.name
    with tempfile.NamedTemporaryFile(mode='w', suffix='.R', delete=False) as f:
        f.write(R_FIT_SCRIPT)
        r_path = f.name
    try:
        result = subprocess.run(
            ['Rscript', r_path, csv_path, dist, formula],
            capture_output=True, text=True, timeout=120
        )
        lines = [l.strip() for l in result.stdout.splitlines() if l.strip()]
        return json.loads(lines[-1]) if lines else {"success": False}
    except Exception as e:
        return {"success": False, "error": str(e)}
    finally:
        Path(csv_path).unlink(missing_ok=True)
        Path(r_path).unlink(missing_ok=True)

def generate_data(dist, n, seed=42):
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(0, 1, n)
    if dist in ["NO", "LOGNO"]:
        y = np.exp(1 + x1) + rng.normal(0, 0.5, n) if dist == "LOGNO" else 2 + x1 + rng.normal(0, 0.5, n)
        y = np.abs(y) + 0.01
    elif dist == "GA":
        mu = np.exp(1 + x1)
        y = rng.gamma(shape=4, scale=mu/4)
    elif dist == "PO":
        y = rng.poisson(np.exp(1 + x1)).astype(float)
    elif dist == "NBI":
        mu = np.exp(1 + x1)
        y = rng.negative_binomial(2, 2/(2+mu)).astype(float)
    elif dist == "BE":
        mu = 1 / (1 + np.exp(-(0.5 + x1)))
        y = np.clip(rng.beta(mu * 5, (1 - mu) * 5), 0.01, 0.99)
    elif dist == "ZIP":
        mu = np.exp(0.5 + x1)
        base = rng.poisson(mu)
        y = np.where(rng.uniform(0, 1, n) < 0.2, 0, base).astype(float)
    elif dist == "ZAGA":
        mu = np.exp(0.5 + x1)
        base = rng.gamma(2, mu/2)
        y = np.where(rng.uniform(0, 1, n) < 0.2, 0.0, base)
    elif dist == "BI":
        mu = 1 / (1 + np.exp(-(0.5 + x1)))
        y = rng.binomial(10, mu).astype(float)
    else:
        y = rng.normal(0, 1, n)
    return {"y": y, "x1": x1}

print(f"Benchmark config / 基准测试配置:")
print(f"  Distributions / 分布: {DISTRIBUTIONS}")
print(f"  Data sizes / 数据规模: {DATA_SIZES}")
print(f"  Repeats / 重复次数: {N_REPEATS}")
print(f"  Total tests / 总测试数: {len(DISTRIBUTIONS) * len(DATA_SIZES)}")

## 4. Run Benchmark

In [ ]:
# Run the full benchmark / 运行完整基准测试
# This may take several minutes / 这可能需要几分钟
bench_results = []
total_tests = len(DISTRIBUTIONS) * len(DATA_SIZES)
test_num = 0

for dist in DISTRIBUTIONS:
    for n in DATA_SIZES:
        test_num += 1
        print(f"[{test_num}/{total_tests}] {dist} n={n}...", end=" ")
        
        data = generate_data(dist, n)
        formula = "y ~ x1"
        
        # Python timing / Python 计时
        py_times = []
        py_ok = True
        for _ in range(N_REPEATS):
            try:
                t0 = time.time()
                model = gamlss(formula, family=resolve_family(dist), data=data, verbose=False)
                py_times.append(time.time() - t0)
            except Exception as e:
                py_ok = False
                break
        
        py_time = np.median(py_times) if py_ok and py_times else np.nan
        
        # R timing / R 计时
        r_result = run_r_fit(dist, formula, data)
        r_time = r_result.get("r_time", np.nan) if r_result.get("success") else np.nan
        
        speedup = r_time / py_time if not np.isnan(r_time) and not np.isnan(py_time) and py_time > 0 else np.nan
        
        bench_results.append({
            "dist": dist,
            "n": n,
            "py_time": py_time,
            "r_time": r_time,
            "speedup": speedup
        })
        
        speedup_str = f"{speedup:.1f}x" if not np.isnan(speedup) else "N/A"
        py_str = f"{py_time:.3f}s" if not np.isnan(py_time) else "FAIL"
        r_str = f"{r_time:.3f}s" if not np.isnan(r_time) else "FAIL"
        print(f"Python={py_str}, R={r_str}, speedup={speedup_str}")

print("\n✓ Benchmark complete / 基准测试完成")

## 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

df = pd.DataFrame(bench_results)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Bar chart: speedup by distribution (averaged over sizes)
# 1. 柱状图：各分布的平均加速比
speedup_by_dist = df.groupby('dist')['speedup'].mean().reindex(DISTRIBUTIONS)
colors = ['#2196F3' if s >= 1 else '#FF5722' for s in speedup_by_dist.fillna(0)]
bars = axes[0].bar(speedup_by_dist.index, speedup_by_dist.values, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0].axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='1x (no speedup)')
axes[0].set_xlabel('Distribution / 分布', fontsize=11)
axes[0].set_ylabel('Speedup (R time / Python time)', fontsize=11)
axes[0].set_title('Average Speedup by Distribution\n各分布平均加速比', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, speedup_by_dist.values):
    if not np.isnan(val):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.1f}x', ha='center', va='bottom', fontsize=8)

# 2. Line chart: Python time vs data size
# 2. 折线图：Python 时间 vs 数据规模
for dist in DISTRIBUTIONS:
    dist_data = df[df['dist'] == dist].sort_values('n')
    axes[1].plot(dist_data['n'], dist_data['py_time'], marker='o', linewidth=1.5,
                 markersize=5, label=dist, alpha=0.8)
axes[1].set_xlabel('Data size (n) / 数据规模', fontsize=11)
axes[1].set_ylabel('Python time (s) / Python 时间（秒）', fontsize=11)
axes[1].set_title('Python Fitting Time vs Data Size\nPython 拟合时间 vs 数据规模', fontsize=12)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

# 3. Heatmap: speedup by (distribution, data size)
# 3. 热力图：（分布，数据规模）的加速比
pivot = df.pivot(index='dist', columns='n', values='speedup').reindex(DISTRIBUTIONS)
im = axes[2].imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=20)
axes[2].set_xticks(range(len(DATA_SIZES)))
axes[2].set_xticklabels([str(n) for n in DATA_SIZES])
axes[2].set_yticks(range(len(DISTRIBUTIONS)))
axes[2].set_yticklabels(DISTRIBUTIONS)
axes[2].set_xlabel('Data size (n) / 数据规模', fontsize=11)
axes[2].set_ylabel('Distribution / 分布', fontsize=11)
axes[2].set_title('Speedup Heatmap (R/Python)\n加速比热力图', fontsize=12)
plt.colorbar(im, ax=axes[2], label='Speedup')
for i in range(len(DISTRIBUTIONS)):
    for j in range(len(DATA_SIZES)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            axes[2].text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8,
                         color='black' if val < 15 else 'white')

plt.suptitle('OmniLSS CPU Performance Benchmark\nOmniLSS CPU 性能基准测试',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("✓ Visualization complete / 可视化完成")

## 6. Summary Table

In [ ]:
# Full results table / 完整结果表
print("\n=== Full Benchmark Results / 完整基准测试结果 ===")
print(f"{'Dist':<8} {'n':>7} {'Python(s)':>10} {'R(s)':>10} {'Speedup':>10}")
print("-" * 50)
for r in bench_results:
    py_str = f"{r['py_time']:.4f}" if not np.isnan(r['py_time']) else "FAIL"
    r_str = f"{r['r_time']:.4f}" if not np.isnan(r['r_time']) else "FAIL"
    sp_str = f"{r['speedup']:.2f}x" if not np.isnan(r['speedup']) else "N/A"
    print(f"{r['dist']:<8} {r['n']:>7} {py_str:>10} {r_str:>10} {sp_str:>10}")

# Summary statistics / 汇总统计
valid_speedups = [r['speedup'] for r in bench_results if not np.isnan(r['speedup'])]
print("\n=== Speedup Statistics / 加速比统计 ===")
print(f"Mean speedup / 平均加速比:   {np.mean(valid_speedups):.2f}x")
print(f"Median speedup / 中位加速比: {np.median(valid_speedups):.2f}x")
print(f"Max speedup / 最大加速比:    {np.max(valid_speedups):.2f}x")
print(f"Min speedup / 最小加速比:    {np.min(valid_speedups):.2f}x")
print(f"Tests with speedup > 1x:     {sum(1 for s in valid_speedups if s > 1)}/{len(valid_speedups)}")

## Summary
This notebook benchmarked OmniLSS CPU performance against R GAMLSS across 9 distributions and 3 data sizes.

 notebook  OmniLSS  CPU  R GAMLSS  9  3 

### Key Findings
- **Overall speedup**: OmniLSS is typically 5-30x faster than R GAMLSS on CPU
- **Scaling**: Python scales better with data size due to JAX JIT compilation
- **Distribution variation**: Speedup varies by distribution complexity

- ****OmniLSS  CPU  R GAMLSS  5-30 
- **** JAX JIT Python 
- ****

---

**Related Notebooks /  Notebooks**:
- [06_performance_gpu.ipynb](06_performance_gpu.ipynb) - GPU performance / GPU 
- [07_performance_tpu.ipynb](07_performance_tpu.ipynb) - TPU performance / TPU 
- [08_comprehensive_comparison.ipynb](08_comprehensive_comparison.ipynb) - Full comparison